# DINOv3 — one backbone, many outputs

**DINOv3** (Meta AI, 2025) is a *self-supervised* vision foundation model. It is
trained on billions of images **without any labels**, and the result is a Vision
Transformer that turns any image into rich, general-purpose **features**.

The key idea: you almost never *train* DINOv3. You **freeze** it, extract its
features once, and attach a tiny task-specific head (a linear layer, a k-NN, a
clustering step, or a small decoder). The same frozen features are strong at
classification, retrieval, segmentation, depth, tracking, correspondence and more.

This notebook is a practical map of **what DINOv3 outputs** and **which output you
use for which application** — light on internals, focused on usage.

## The mental model: `image → features → tiny head`

```
                ┌────────────────────────┐
   image  ──▶   │   DINOv3 (frozen)      │  ──▶   features
                └────────────────────────┘           │
                                                      ├─▶ CLS token   (1 vector / image)
                                                      ├─▶ patch tokens (grid of vectors)
                                                      └─▶ register tokens (housekeeping)
```

* You send in an image.
* DINOv3 returns a set of embedding vectors.
* You pick the **right output** and bolt on a lightweight head for your task.

Because the backbone is frozen, this is cheap: features can be pre-computed and
cached, and heads train in seconds/minutes on a CPU or a single GPU.

## Setup

DINOv3 is served through 🤗 `transformers`. The pretrained weights are **gated**
on the Hugging Face Hub — accept the license on the model page once, then log in
with `huggingface-cli login` (or set `HF_TOKEN`).

In [ ]:
# One-time install (uncomment to run)
# %pip install -U "transformers>=4.53" torch torchvision pillow scikit-learn matplotlib numpy requests

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
import numpy as np
from PIL import Image
import requests
from io import BytesIO

def load_image(path_or_url, size=768):
    '''Load a local path OR a URL into a RGB PIL image.'''
    if str(path_or_url).startswith("http"):
        img = Image.open(BytesIO(requests.get(path_or_url, timeout=30).content))
    else:
        img = Image.open(path_or_url)
    return img.convert("RGB")

# A couple of sample images (replace with your own paths/URLs)
URL_A = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats
URL_B = "http://images.cocodataset.org/val2017/000000000139.jpg"  # a room
image = load_image(URL_A)
image


## The three outputs of the backbone

Run an image through a DINOv3 *backbone* (a `...-pretrain-...` checkpoint) and the
`last_hidden_state` is a sequence of token vectors laid out as:

```
[ CLS ] [ register tokens ] [ patch_0  patch_1  ...  patch_N ]
```

| Output | Shape (per image) | What it is | Use it for |
|---|---|---|---|
| **CLS token** | `(D,)` — one vector | A **global summary** of the whole image | classification, retrieval, similarity, clustering, dedup |
| **Patch tokens** | `(H/16 · W/16, D)` — a grid | A **local** vector per image patch | segmentation, depth, correspondence, matching, tracking, detection |
| **Register tokens** | a few vectors | Internal "scratch" tokens that soak up artifacts | usually **ignored** downstream |

`D` (the embedding size) depends on the variant: 384 (S) → 768 (B) → 1024 (L) →
1280+ (H) → 4096 (7B). Patch size is **16**, so a 768×768 image gives a 48×48 grid.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

# Pick any backbone. ViT-B is a great CPU-friendly default.
BACKBONE = "facebook/dinov3-vitb16-pretrain-lvd1689m"

processor = AutoImageProcessor.from_pretrained(BACKBONE)
model = AutoModel.from_pretrained(BACKBONE).to(DEVICE).eval()

@torch.inference_mode()
def extract(img):
    '''Return (cls_vector, patch_grid) for one PIL image.
       cls   : (D,)          global embedding
       grid  : (gh, gw, D)   dense per-patch embeddings
    '''
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    out = model(**inputs)
    hidden = out.last_hidden_state[0]              # (num_tokens, D)

    _, _, H, W = inputs["pixel_values"].shape
    gh, gw = H // model.config.patch_size, W // model.config.patch_size
    n_patches = gh * gw

    cls   = hidden[0]                              # first token = CLS
    patch = hidden[-n_patches:]                    # last N tokens = patches
    grid  = patch.reshape(gh, gw, -1)
    return cls.cpu().numpy(), grid.cpu().numpy()

cls, grid = extract(image)
print("CLS embedding :", cls.shape)      # (D,)
print("patch grid    :", grid.shape)     # (gh, gw, D)


## Scenario 1 — Global embedding → similarity, retrieval, classification

The **CLS token** is a single fingerprint for the whole image. Cosine similarity
between two CLS vectors measures how alike two images are — the basis of
**image search / retrieval**, **near-duplicate detection**, and **clustering**.
Stack many CLS vectors + labels and a one-layer classifier ("linear probe") gives
strong classification without fine-tuning the backbone.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

cls_a, _ = extract(load_image(URL_A))   # cats
cls_b, _ = extract(load_image(URL_B))   # room

print("similarity(cats, cats):", round(cosine(cls_a, cls_a), 3))
print("similarity(cats, room):", round(cosine(cls_a, cls_b), 3))

# Retrieval sketch: embed a gallery once, then rank by cosine to a query.
# gallery = np.stack([extract(load_image(p))[0] for p in gallery_paths])
# scores  = gallery @ query_cls / (norm(gallery,axis=1)*norm(query_cls))
# top_k   = scores.argsort()[::-1][:5]


## Scenario 2 — Dense patch features → PCA "feature image"

The classic DINO trick: run **PCA** on the patch grid and map the top-3
components to RGB. Objects and their parts light up in consistent colors, and the
foreground pops out of the background — a label-free look at what the model
"sees". Great for a sanity check and for unsupervised object/part discovery.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

_, grid = extract(image)
gh, gw, D = grid.shape
flat = grid.reshape(-1, D)

pca = PCA(n_components=3).fit_transform(flat)
pca = (pca - pca.min(0)) / (pca.ptp(0) + 1e-8)        # normalize to [0,1]
feat_img = pca.reshape(gh, gw, 3)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(image); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(feat_img); ax[1].set_title("PCA of patch features"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## Scenario 3 — Patch features → unsupervised segmentation

Cluster the patch vectors (e.g. **k-means**) and you get a segmentation map with
**no labels and no training** — pixels with similar semantics fall in the same
cluster. Swap k-means for a small trained linear layer over the patch features and
you get supervised **semantic segmentation** (this is exactly how the released
segmentation heads work).

In [ ]:
from sklearn.cluster import KMeans

_, grid = extract(image)
gh, gw, D = grid.shape

K = 5
labels = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(grid.reshape(-1, D))
seg = labels.reshape(gh, gw)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(image); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(seg, cmap="tab10"); ax[1].set_title(f"k-means over patches (K={K})"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## Scenario 4 — Patch features → dense correspondence & matching

Patch embeddings are comparable **across images**: the same object part gets a
similar vector in different photos. For each patch in image A, the nearest patch
(by cosine) in image B is its match. This powers **correspondence, feature
matching, one-shot part transfer, and video point tracking** — no keypoint
detector or matcher trained on pairs required.

In [ ]:
def best_match(gridA, gridB, ay, ax_):
    '''Given a patch (ay,ax_) in A, return the best-matching (y,x) in B.'''
    a = gridA[ay, ax_]                              # (D,)
    B = gridB.reshape(-1, gridB.shape[-1])          # (Nb, D)
    a = a / (np.linalg.norm(a) + 1e-8)
    Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    idx = (Bn @ a).argmax()
    return divmod(int(idx), gridB.shape[1])         # (y, x) in B's grid

_, gridA = extract(load_image(URL_A))
_, gridB = extract(load_image(URL_A))               # same scene here as a demo
y, x = best_match(gridA, gridB, gridA.shape[0] // 2, gridA.shape[1] // 2)
print("center patch of A best matches patch", (y, x), "in B")


## Scenario 5 — Linear probe (train a tiny classifier on frozen features)

The standard way to *use* DINOv3 for a labeled task: freeze the backbone, cache
CLS embeddings for your dataset, and fit a single `Linear` (or logistic
regression / k-NN). Minutes to train, strong accuracy, tiny to deploy.

In [ ]:
import torch.nn as nn

# Pretend we cached CLS features + labels for a small dataset:
# X = np.stack([extract(load_image(p))[0] for p in paths])   # (N, D)
# y = np.array(labels)                                        # (N,)

D = model.config.hidden_size
num_classes = 10
head = nn.Linear(D, num_classes)     # <-- the ONLY thing you train

# opt = torch.optim.AdamW(head.parameters(), lr=1e-3)
# for epoch in range(50):
#     logits = head(torch.tensor(X).float())
#     loss = nn.functional.cross_entropy(logits, torch.tensor(y))
#     opt.zero_grad(); loss.backward(); opt.step()
print("Trainable params in the head:", sum(p.numel() for p in head.parameters()))


## Scenario 6 — Ready-made task head: monocular depth

Meta also ships **checkpoints with heads already attached**, so you get task
outputs directly. Here is a DPT depth head on a DINOv3 backbone — feed an image,
get a per-pixel depth map. (This was the original cell in this notebook.)

In [ ]:
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

depth_id = "facebook/dinov3-vitl16-dpt-lvd1689m"   # DINOv3 backbone + DPT depth head
processor_d = AutoImageProcessor.from_pretrained(depth_id)
model_d = AutoModelForDepthEstimation.from_pretrained(depth_id).to(DEVICE).eval()

img = load_image(URL_B)
inputs = processor_d(images=img, return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    outputs = model_d(**inputs)

depth = processor_d.post_process_depth_estimation(
    outputs, target_sizes=[(img.height, img.width)]
)[0]["predicted_depth"]

plt.figure(figsize=(5, 4))
plt.imshow(depth.cpu().numpy(), cmap="inferno"); plt.axis("off"); plt.title("depth"); plt.show()


## Scenario 7 — Domain-specialized checkpoints

DINOv3 also ships backbones trained on **satellite/aerial imagery** (`...-sat493m`)
and heads for domain tasks such as **canopy-height / dense regression**
(e.g. `facebook/dinov3-vitl16-chmv2-dpt-head`). Same recipe — swap the checkpoint
name, keep the `features → head` pattern — but the features are tuned for remote
sensing instead of everyday photos.

```python
# Aerial / satellite backbone
sat = AutoModel.from_pretrained("facebook/dinov3-vitl16-pretrain-sat493m")
```


## Choosing a variant (the model zoo)

| Checkpoint (`facebook/dinov3-…`) | Params | Emb `D` | When to pick it |
|---|---|---|---|
| `vits16-pretrain-lvd1689m` | 21 M | 384 | fastest, edge/CPU, prototyping |
| `vitb16-pretrain-lvd1689m` | 86 M | 768 | **balanced default** |
| `vitl16-pretrain-lvd1689m` | 300 M | 1024 | high quality, single GPU |
| `vith16plus-pretrain-lvd1689m` | 840 M | 1280+ | top accuracy |
| `vit7b16-pretrain-lvd1689m` | 6.7 B | 4096 | research SOTA, multi-GPU |
| `convnext-{tiny…large}-…` | — | — | CNN backbone (easier to deploy/export) |
| `…-pretrain-sat493m` | — | — | satellite / aerial imagery |

Rule of thumb: **start with ViT-B**, move up only if the metric demands it. Larger
models mostly buy you sharper dense features and better fine-grained retrieval.

## Cheat-sheet: which output for which task

| You want to… | Use | Head you add |
|---|---|---|
| Classify whole images | **CLS token** | linear / logistic / k-NN |
| Search / retrieve / dedup | **CLS token** | cosine similarity, ANN index |
| Cluster an image collection | **CLS token** | k-means |
| Semantic segmentation | **patch tokens** | linear layer per patch (or k-means, unsupervised) |
| Depth / dense regression | **patch tokens** | DPT head (or use a `-dpt-` checkpoint) |
| Object / part discovery | **patch tokens** | PCA / clustering |
| Matching, correspondence, tracking | **patch tokens** | nearest-neighbor in feature space |
| Detection | **patch tokens** | detection head on the feature grid |

**One sentence to remember:** DINOv3 gives you a *global* vector (CLS) for
image-level tasks and a *grid* of local vectors (patches) for pixel-level tasks —
freeze the backbone, cache the features, and train only a tiny head.